In [14]:
import os
from langchain.agents import create_agent
from langchain.tools import tool


In [1]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')
os.environ['LANGSMITH_PROJECT'] = 'basic-agent'

In [ ]:

import requests



@tool
def get_weather(location: str) -> str:
    """Get real-time weather information for any location or location worldwide."""

    # Step 1: Find coordinates using geocoding
    geo_url = "https://geocoding-api.open-meteo.com/v1/search"

    geo_params = {
        "name": location,
        "count": 5,
        "language": "en",
        "format": "json"
    }

    try:
        geo_response = requests.get(
            geo_url, params=geo_params, timeout=10
        )
        geo_response.raise_for_status()

        locations = geo_response.json().get("results", [])

        if not locations:
            return f"Location '{location}' not found. Please try another location."

        # Select the best matching location
        location = locations[0]

        latitude = location["latitude"]
        longitude = location["longitude"]
        name = location["name"]
        country = location.get("country", "Unknown")
        timezone = location.get("timezone", "auto")

        # Step 2: Get current weather
        weather_url = "https://api.open-meteo.com/v1/forecast"

        weather_params = {
            "latitude": latitude,
            "longitude": longitude,
            "current": [
                "temperature_2m",
                "relative_humidity_2m",
                "apparent_temperature",
                "precipitation",
                "weather_code",
                "wind_speed_10m"
            ],
            "timezone": timezone
        }

        weather_response = requests.get(
            weather_url, params=weather_params, timeout=10
        )
        weather_response.raise_for_status()

        weather = weather_response.json()["current"]

        return f"""
        Weather Report for {name}, {country}

        Local Time: {weather['time']}
        Temperature: {weather['temperature_2m']} °C
        Feels Like: {weather['apparent_temperature']} °C
        Humidity: {weather['relative_humidity_2m']}%
        Precipitation: {weather['precipitation']} mm
        Wind Speed: {weather['wind_speed_10m']} km/h
        Weather Code: {weather['weather_code']}
        """

    except requests.RequestException as e:
        return f"Weather service error: {e}"







In [20]:
# Step 3: Create agent
agent = create_agent(
    model="groq:openai/gpt-oss-20b",
    tools=[get_weather],
    system_prompt="""
    You are a helpful weather assistant.

    When users ask about weather:
    1. Identify the location.
    2. Call get_weather with the location.
    3. Explain the returned weather information clearly.
    4. Never invent weather information.
    """
)


In [21]:
# Step 4: Test agent
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What's the current weather in Tokyo?"
        }
    ]
})

print(response["messages"][-1].content)

Here’s the latest weather snapshot for **Tokyo, Japan** (as of 17:00 local time on 24 Nov 2026):

| Item | Value |
|------|-------|
| **Temperature** | 15.8 °C |
| **Feels Like** | 16.0 °C |
| **Humidity** | 91 % |
| **Precipitation** | 1.3 mm (light rain) |
| **Wind Speed** | 8.7 km/h (gentle breeze) |
| **Weather Code** | 63 (light rain) |

**What this means for you:**
- It’s cool and damp, so you’ll want a light jacket or sweater if you’re heading out.
- The humidity is high, which can make the air feel a bit muggy.
- Light rain is falling, so a small umbrella or waterproof jacket would be handy.
- Winds are mild, so they won’t be a major factor.

Enjoy your day in Tokyo! If you need weather updates for another location or a different time, just let me know.


In [22]:
agent.invoke({"messages":{"role":"user","content":"What is the weather in Butwal ?"}})

{'messages': [HumanMessage(content='What is the weather in Butwal ?', additional_kwargs={}, response_metadata={}, id='7ccd0b24-fa5b-4cf8-9a91-329002079cab'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to identify location: Butwal. Then call get_weather function with city: "Butwal". Then explain returned weather info. Use the function.', 'tool_calls': [{'id': 'fc_0861bfa7-5f5e-46ad-9a2d-627069341ec8', 'function': {'arguments': '{"city":"Butwal"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 56, 'prompt_tokens': 191, 'total_tokens': 247, 'completion_time': 0.057336714, 'completion_tokens_details': {'reasoning_tokens': 32}, 'prompt_time': 0.009275879, 'prompt_tokens_details': None, 'queue_time': 0.286224682, 'total_time': 0.066612593}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_9b8528b477', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider